In [1]:
# This program will extract individual number from a image file and create separate image file for each number.
# This program needs OpenCV library, it can be installed as 
#                 pip install opencv-python

import cv2
import numpy as np

def extract_digits_perfectly(image_path):
    # 1. Load the image
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Could not load image. Check the file path.")
        return
        
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 2. Threshold the image to turn black squares white and white background black
    _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY_INV)
    
    # 3. Find the outlines (contours) of the squares
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 4. Filter contours to ensure we only select the digit boxes
    digit_boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h
        # Only keep contours that look like a digit box (size filtering)
        if area > 1000 and w > 50 and h > 50:
            digit_boxes.append((x, y, w, h))
            
    print(f"Found {len(digit_boxes)} digit boxes.")
    
    # 5. Sort the boxes so they go from 0 to 9 (top row left-to-right, then bottom row)
    # Sort by Y coordinate first to separate the two rows
    digit_boxes.sort(key=lambda box: box[1])
    
    # Separate into top row (first 5) and bottom row (last 5) and sort each by X coordinate
    top_row = sorted(digit_boxes[:5], key=lambda box: box[0])
    bottom_row = sorted(digit_boxes[5:], key=lambda box: box[0])
    final_sorted_boxes = top_row + bottom_row
    
    # 6. Crop and save each box precisely
    for idx, (x, y, w, h) in enumerate(final_sorted_boxes):
        cropped = img[y:y+h, x:x+w]
        filename = f"sample_data/images/digit_{idx}.jpg"
        cv2.imwrite(filename, cropped)
        print(f"Saved {filename} accurately.")

# To run this, save your image as 'digits.png' in the same folder and uncomment below:
extract_digits_perfectly('sample_data/images/Numbers.jpg')